# Exercise 0 — What is a Tensor?
A tensor is a multi-dimensional array that PyTorch uses to store data and perform mathematical operations on the GPU.

---

# Exercise 1 — Why do we need all five steps?
Prediction gives an output, loss measures error, gradients show how to change parameters, the optimizer updates them, and gradient reset prevents updates from stacking incorrectly.

### Solution — What would happen if we made predictions but never calculated a loss?

Without a loss function, the model receives no feedback about how wrong its predictions are.  
It can produce outputs, but it has no signal telling it how to improve.  
Training would not progress, and the model would never learn.

### Solution — What if we calculated gradients but never updated the parameters?

Gradients describe how the parameters should change, but if we never apply them, the model stays exactly the same.  
The loss will not decrease, and the model will not improve over time.  
Training becomes a static loop with no learning.

### Solution — What if we updated parameters but never reset the gradients?

If gradients are not reset, they accumulate across iterations.  
This causes updates to become larger and larger, often leading to unstable training or divergence.  
Resetting gradients ensures each update is based only on the current batch.

---

# Exercise 2 — Batches and Epochs

A **batch** is a small subset of the dataset that the model processes in one training iteration.  
An **epoch** means that the entire dataset has passed through the neural network once. Every sample (batch) has been used to make predictions, compute loss, calculate gradients, and update the model’s parameters. Because a single pass is rarely enough for the model to learn meaningful patterns, training typically involves many epochs.

Training on the entire dataset at once would be slow and require too much memory. Using batches makes training more efficient by reducing memory usage and allowing the model to update its parameters more frequently. The small amount of noise introduced by batch-based gradients often improves generalization, helping the model learn better overall.

 ---
# Exercise 3 - Why is the default datatype for tensors float32?

Neural networks use float32 because the entire learning process is based on continuous values.
The loss function is continuous (even for binary labels), and the gradients computed from this loss are also continuous.
During training, the weights are updated in small decimal steps, which requires a floating‑point datatype capable of representing these small changes.

Integers cannot store decimals and would therefore prevent both gradient calculations and weight updates.
For this reason, training data and model parameters must use a floating‑point format such as float32.

 ---
# Exercise 4 – Loss, Gradients and Backpropagation

The **loss** is a single value that measures how wrong the model’s predictions are.  
The **gradient** is the derivative of the loss with respect to each parameter and shows how the parameters should change to reduce the loss.  
**Backpropagation** is the algorithm that computes these gradients by applying the chain rule backward through the network.  
The **optimizer** uses the gradients to update the parameters, completing the learning loop.

 ---
# Exercise 5 – Dot-product influence on neural networks

The dot-product requires that the number of columns in the input tensor matches the number of rows in the weight tensor. Because of this, each layer transforms the input into a new tensor with a new shape determined by the weight matrix. This is how neural networks change the “resolution” or dimensionality of the data as it moves deeper through the layers. In other words, the dot-product rule controls how information is transferred and reshaped from one layer to the next.

 ---
# Exercise 6 – The Argmax function

When you use dim = 0 in a reduction operation such as argmax(), PyTorch collapses the tensor vertically. This means it compares the values column by column instead of row by row. For each column, PyTorch looks at all rows and returns the index of the row that contains the highest value in that column. The output is therefore a vector where each element represents the position of the maximum value within its respective column.

 ---
# Exercise 8 – The @ operator

The  @ operator  performs matrix multiplication between two tensors. It takes the rows of the first tensor and multiplies them with the columns of the second tensor to produce a new tensor containing the combined weighted values.

 ---
# Exercise 9 - Tensor Shapes and Broadcasting Logic

- X @ W has shape (100, 1) because (100,1) @ (1,1) → (100,1).
- Adding b works because b has shape (1,), which broadcasts to (100,1).
- If W had shape (1,), matrix multiplication would fail since @ expects 2D shapes.

A shape mismatch happens when inner dimensions don’t align, e.g. (100,2) @ (1,1).
Fix: make W shape (2,1) so multiplication is valid.

Understanding shapes is essential because every layer in a neural network
depends on correct dimensional alignment. Wrong shapes = wrong math.

 ---
# Exercise 10 - Reasoning About Autograd and the Computation Graph

The graph consists of:
X → matmul → z → add b → y_hat → subtract y_true → square → mean → loss.

W and b require gradients; X and y_true typically do not.

PyTorch computes gradients by chaining local derivatives backward from loss
through each operation.

If W.requires_grad=False → no gradient for W.
If backward() is missing → no gradients at all.
If zero_grad() is missing → gradients accumulate and distort updates.

 ---
# Exercise 11 - Understandning the Meaning of the Gradient

A large positive W.grad means increasing W increases the loss.
Gradient descent will therefore decrease W.

A very small gradient means the loss surface is flat → slow learning.
A very large gradient means steep surface → unstable updates.

Small gradients may require a higher learning rate.
Large gradients may require a lower learning rate or clipping.

 ---
# Exercise 12 - Reasoning About the Training Loop

The order matters:

1. Forward pass → you need predictions.
2. Loss → you need an error signal.
3. Backward → you need gradients before updating.
4. Update → uses the gradients.
5. Zero_grad → prepares for the next iteration.

Updating must be inside torch.no_grad() so the update itself is not tracked
as part of the computation graph.

Gradients accumulate by default so multiple backward passes can be combined,
but in a simple loop they must be cleared each iteration.

 ---
# Exercise 5 - Convergence, Stability and Learning Dynamics

Convergence means the loss stops improving and parameter updates become small.

The model may not reach exact true parameters because:
- Data contains noise
- Learning rate is not ideal
- Numerical precision or model limitations

Larger learning rate → faster but riskier.
Smaller learning rate → slower but stable.
Noisy data → imperfect convergence.
Bad initialization → slower or unstable learning.

Underfitting: high loss everywhere.
Overfitting: low training loss, high validation loss.
Correct learning: both losses decrease and stabilize together.
